In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2025)

# ---------- Helper functions ----------

def two_stage_bootstrap_diff(Xa, Xb, B=10000, rng=None):
    nA, kA = Xa.shape
    nB, kB = Xb.shape
    if kA != kB:
        raise ValueError("Both groups must have the same number of questions (columns).")
    k = kA
    if rng is None:
        rng = np.random.default_rng()
    diffs = np.empty(B)
    for b in range(B):
        # resample participants
        idxA = rng.integers(0, nA, size=nA)
        print("Resampled participants for Group A:", idxA)
        idxB = rng.integers(0, nB, size=nB)
        print("Resampled participants for Group B:", idxB)
        Xa_b = Xa[idxA, :]
        print("Resampled data for Group A:\n", Xa_b)
        Xb_b = Xb[idxB, :]
        print("Resampled data for Group B:\n", Xb_b)
        # resample questions (same columns for both groups)
        idxQ = rng.integers(0, k, size=k)
        Xa_b = Xa_b[:, idxQ]
        Xb_b = Xb_b[:, idxQ]
        diffs[b] = Xa_b.mean() - Xb_b.mean()
        exit(1)
    return diffs

def bootstrap_pvalue_two_stage(Xa, Xb, B=10000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
        
    # Observed statistic
    obs_diff = Xa.mean() - Xb.mean()
    
    # 1) CI from the ordinary (non-null) two-stage bootstrap
    diffs = two_stage_bootstrap_diff(Xa, Xb, B=B, rng=rng)
    ci95 = np.percentile(diffs, [2.5, 97.5])
    se = diffs.std(ddof=1)

    # 2) Null-imposing bootstrap for p-value
    muA = Xa.mean()
    muB = Xb.mean()
    mu_pool = (muA + muB) / 2.0
    Xa0 = Xa - muA + mu_pool
    Xb0 = Xb - muB + mu_pool

    diffs0 = two_stage_bootstrap_diff(Xa0, Xb0, B=B, rng=rng)
    
    # two-sided and one-sided p-values
    p_two_sided = np.mean(np.abs(diffs0) >= abs(obs_diff))
    p_right = np.mean(diffs0 >= obs_diff)  # H1: A > B
    p_left  = np.mean(diffs0 <= obs_diff)  # H1: A < B
    
    return {
        "obs_diff": obs_diff,
        "se": se,
        "ci95": ci95,
        "p_two_sided": p_two_sided,
        "p_right": p_right,
        "p_left": p_left,
        "boot_dist": diffs,
        "null_dist": diffs0,
    }

# ---------- Main script ----------

# Load CSV files (no headers assumed, adjust if yours have headers)
Xa = pd.read_csv("human_t1_stable.csv", header=None).to_numpy()
Xb = pd.read_csv("model_t1_stable.csv", header=None).to_numpy()

# excel output
output_file = "task3_id.xlsx"

# Participant-level means
Xa_means = Xa.mean(axis=1)   # each participant mean across questions
Xb_means = Xb.mean(axis=1)

# Group-level means
meanA = Xa_means.mean()
meanB = Xb_means.mean()

# Run bootstrap test
results = bootstrap_pvalue_two_stage(Xa, Xb, B=10000, rng=rng)

# Print results
print("Group A mean:", meanA)
print("Group B mean:", meanB)
print("Observed difference (A - B):", results["obs_diff"])
print("Bootstrap SE:", results["se"])
print("95% CI:", results["ci95"])
print("Two-sided p-value:", results["p_two_sided"])
print("One-sided p (A > B):", results["p_right"])
print("One-sided p (A < B):", results["p_left"])

# ---------- Visualization ----------

# 1. Bar chart of group means
plt.figure(figsize=(6,4))
plt.bar(["Group A","Group B"], [meanA, meanB], color=["skyblue","salmon"], alpha=0.8)
plt.ylabel("Mean Score")
plt.title("Group Means")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

# 2. Histogram of bootstrap distribution of difference
plt.figure(figsize=(7,5))
plt.hist(results["boot_dist"], bins=40, color="gray", alpha=0.7, density=True)
plt.axvline(results["obs_diff"], color="red", linestyle="--", label=f"Observed diff = {results['obs_diff']:.3f}")
plt.axvline(results["ci95"][0], color="blue", linestyle=":", label="95% CI bounds")
plt.axvline(results["ci95"][1], color="blue", linestyle=":")
plt.xlabel("Bootstrap Difference (A - B)")
plt.ylabel("Density")
plt.title("Bootstrap Distribution of Difference")
plt.legend()
plt.show()

output = {
    "Group A mean": meanA,
    "Group B mean": meanB,
    "Observed difference (A - B)": results["obs_diff"],
    "Bootstrap SE": results["se"],
    "95% CI": results["ci95"],
    "Two-sided p-value": results["p_two_sided"],
    "One-sided p (A > B)": results["p_right"],
    "One-sided p (A < B)": results["p_left"],
    }
# df = pd.DataFrame(output)

# with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
#     df.to_excel(writer, sheet_name="Summary_raw", index=False)

Resampled participants for Group A: [ 6 13 13  5 13 11  8 11 10 13  5  1  6  4]
Resampled participants for Group B: [ 5 11 11  8  7  3  0  4  5  2  9  2]
Resampled data for Group A:
 [[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 0 1]]
Resampled data for Group B:
 [[0 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 1]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 0 1 1 1]
 [0 0 1 0 0]
 [1 1 1 1 1]
 [0 0 0 0 0]
 [1 1 1 0 0]
 [0 0 0 0 0]
 [1 1 1 0 0]]
Resampled participants for Group A: [ 5  3  0 12  6  8  4  0  2  1  2  2  8  1]
Resampled participants for Group B: [ 7  6 10  5  1  3  5  3  3  3  5 10]
Resampled data for Group A:
 [[1 1 1 0 1]
 [1 1 1 0 0]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 0 1]
 [1 1 1 1 1]
 [1 1 1 0 1]]
Resampled data for Group B:
 [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 

KeyboardInterrupt: 